## Genotype Data
- Generates all necessary genotype .csv files

In [1]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer, KNNImputer

In [2]:
fgenotype = "../data/genotype/genotype_matrix.xlsx"

In [3]:
#
# Somehow there are multiple sheets, that I actually could see in libreCalc 
#
excel_p = pd.ExcelFile(fgenotype)
excel_p.sheet_names

['WinSTAT Commands', 'WinSTAT Trigger', 'Tabelle2']

In [4]:
#
# Just Select with sheet_name
#
df_g = pd.read_excel(excel_p, sheet_name="Tabelle2")
# df_g.head(6)

#### Data Processing

In [5]:
#
# export locational df (position in genome)
#
df_loc = pd.DataFrame(data = {"Marker" : df_g.columns[1:],
                              "Chr"    : [2 for i in range(len(df_g.columns[1:]))],
                              "Pos"    : df_g.iloc[0, 1:]})
df_loc.to_csv("../data/genotype/genotype_location.csv", index=False)
# df_loc.head()

In [6]:
#
# Remove irrelevant rows from df, replace 'u' with nan
# (Somehow, this is the way it is encoded) and make 
# genotypes numeric. 
#
df_nan = df_g.iloc[5:, :]
df_nan.index = [i for i in range(len(df_nan))]
df_nan.replace('u', np.float64("nan"), inplace=True)
df_nan.replace("A", 0, inplace=True)
df_nan.replace("h", 1, inplace=True)
df_nan.replace("B", 2, inplace=True)

#
# Initially, I thought GAPIT handles nan values internally,
# thats why also saved the nan df. 
#
df_nan.to_csv("../data/genotype/genotype_matrix_nan.csv", index=False)
# df_nan

/tmp/ipykernel_215285/275345656.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_nan.replace('u', np.float64("nan"), inplace=True)
/tmp/ipykernel_215285/275345656.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_nan.replace("A", 0, inplace=True)
/tmp/ipykernel_215285/275345656.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_nan.replace("h", 1, inplace=True)
/tmp/ipykernel_215285/275345656.py:11: FutureWarning: Downc

In [7]:
#
# NaN values must be handled in the preprocessing step, but you cannot
# just add a mean genotype, you must use different approaches. My first
# idea was to use the most frequent genotype in the markers column (this
# is called the mode imputation).
#
impute_mode = SimpleImputer(strategy="most_frequent")
df_mode = pd.concat([df_nan.iloc[:, 0], 
                     pd.DataFrame(columns = df_nan.columns[1:],                                 # some tmp df
                                  data = impute_mode.fit_transform(X = df_nan.iloc[:, 1:]))],
                     axis=1)
df_mode.to_csv("../data/genotype/genotype_matrix_mode.csv", index=False)
# df_mode

In [8]:
# After some research, I fount that KNN impute followed
# by an int cast is a standard approach. 
impute_knn = KNNImputer()
df_mode = pd.concat([df_nan.iloc[:, 0], 
                     pd.DataFrame(columns = df_nan.columns[1:],                                         # some tmp df
                                  data = impute_knn.fit_transform(X = df_nan.iloc[:, 1:])).round(0)],
                     axis=1)
df_mode.to_csv("../data/genotype/genotype_matrix_knn.csv", index=False)
# df_mode